In [ ]:
import sys
sys.path.insert(0, "../src")
from TownScapeENV import *
import time

In [ ]:
%load_ext tensorboard

# 1

In [ ]:
obs_shape = (5,5)
env = townScape(obs_shape)
# for i in range(60):
#     ic(env.act_decoder(i))
env.reset()
# display_img(env.render())
done = False
while not done:
    act = env.action_space.sample()
    state, reward, done, _, info = env.step(act,debug= False)
display_img(env.render())
ic(info)
ic(env.home_list)

In [ ]:
_,_,_,img_list,_ = env.simulate(1)
for i in img_list:
    display_img(i)

In [ ]:
env.total_time = 0 
row,col = (1,1)
_,_,_,_,info = env.step(env.act_encoder((row,col)),debug=False)
display_img(env.render())
min_home = env.home_list[0]
ic(info,env.home_list)

In [ ]:
env = townScape()
for i in range(100):
    env.reset()
    done = False
    while not done:
        act = env.action_space.sample()
        state, rew, done, _, info = env.step(act,debug=False)
    if(info['status']=='SQUARE'):
        ic(rew,info)
        display_img(env.render())
    

# 2

In [ ]:
from stable_baselines3.common import env_checker
env2 = townScape()
env_checker.check_env(env2)

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import time

In [ ]:
T = int(time.time())
model_dir = f"../training/PPO-{T}"
log_dir = f"../training/logs/PPO-{T}"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

obs_shape = (10,10)
env = townScape(obs_shape)
# env = DummyVecEnv([lambda: env])
env.reset()
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log = log_dir)

In [ ]:
TIMESTEPS = 2048 #total timesteps per ep
train_eps = 3
for ep in range(1,train_eps+1):
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps = False, tb_log_name = "LOG",progress_bar = False)
model.save(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")
env.close()
del model

In [ ]:
training_log_path = f"{log_dir}/LOG_0"
%tensorboard --logdir {training_log_path}

In [ ]:
#test model
env = townScape((10,10))
# model = PPO.load(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")
model = PPO.load('../models/10x10_200', custom_objects={
    "observation_space": env.observation_space, "action_space": env.action_space,
    "lr_schedule": lambda _: 0.0, "clip_range": lambda _: 0.0})
eps = 3
for ep in range(1,1+eps):
    obs, _ = env.reset(requirements={"map":None, "home":0, "target":0})
    # ic(env.obs)
    done = False
    ic('load')
    while not done:
        action, _ = model.predict(obs) #it returns action, state
        obs, reward, done, _, info = env.step(action, debug = False)
    
    display_img(env.render())
    tf,_,_,_,_ = env.simulate()
    median_of_tf = np.median(tf)
    ic(median_of_tf)
    ic(info)
env.close()

# 3

In [ ]:
_,_,_,sim_list,_ = env.simulate(render=True)
frames = [Image.fromarray(f) for f in sim_list]
bio = BytesIO()
frames[0].save(bio, format='GIF', save_all=True, append_images=frames[1:], duration=250, loop=0)
ipyd.display(ipyd.Image(bio.getvalue(), format='gif'))